# 多谱 → 结构（SpectraLLM）

用户已明确暂不推进；历史结果仅作归档，不再运行。

本 notebook 展示已记录的真实评测结果。模型推理需显式开启，始终不进行训练。

In [1]:
import json, os, sys, subprocess, tempfile
from pathlib import Path
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'openmolspec_bench').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from the cloned OpenMolSpec frontend repository')
task_id = 'multispectral.spectrallm.generation'
example_input = {'ir': '1024 intensities on the documented axis', 'raman': '1024 intensities on the same axis', 'h_nmr_ppm': [1.2, 3.65], 'c_nmr_ppm': [18.2, 58.3], 'prompt_format': "must follow the released model's preprocessing"}
print(json.dumps(example_input, indent=2, ensure_ascii=False))

{
  "ir": "1024 intensities on the documented axis",
  "raman": "1024 intensities on the same axis",
  "h_nmr_ppm": [
    1.2,
    3.65
  ],
  "c_nmr_ppm": [
    18.2,
    58.3
  ],
  "prompt_format": "must follow the released model's preprocessing"
}


## 已记录的评测

保留样本量、输入条件和来源文件；流程执行成功不等于精度达标。

In [2]:
index = json.loads((root / 'documentation/assets/evaluation_index.json').read_text())
recorded = index.get(task_id, [])
print(json.dumps(recorded or {'status': 'not_evaluated', 'reason': 'Required compatible assets or adapter unavailable'}, indent=2, ensure_ascii=False))

{
  "status": "not_evaluated",
  "reason": "Required compatible assets or adapter unavailable"
}


## 可选：冻结权重推理

设置 `OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE=1` 并配置工作区和模型环境。缺少权重时跳过或报错，不会启动训练。

In [3]:
if os.environ.get('OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE') == '1':
    runtime = os.environ.get('OPENMOLSPEC_PYTHON', sys.executable)
    workspace = os.environ.get('OPENMOLSPEC_BENCH_ROOT', str(root.parent))
    with tempfile.TemporaryDirectory(prefix='openmolspec-notebook-') as directory:
        command = [runtime, '-m', 'openmolspec_bench', 'run', '--workspace', workspace, '--tasks', task_id, '--limit', '2', '--output', str(Path(directory) / 'result.json')]
        subprocess.run(command, cwd=root, check=True)
        print((Path(directory) / 'result.json').read_text())
else:
    print('Model execution disabled; recorded evaluation displayed above. No training performed.')

Model execution disabled; recorded evaluation displayed above. No training performed.
